# Example 12.5 / 1: Benders on a Two-Scenario Instance

Companion notebook for **12.5 Decomposition** in the AMPL lab Documentation, section "Benders on a Two-Scenario Instance". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Benders on a Two-Scenario Instance

A plant builds capacity $x$ (`Build` in the model) at cost 1 per unit, up to 10 units. Demand is 4 or 8 with
probability 0.5 each, and any shortfall is outsourced at cost 3 per unit. The recourse cost is
$Q_s(x)=3\max(0,d_s-x)$, so the total cost $x+1.5\max(0,4-x)+1.5\max(0,8-x)$ is lowest at
$x=8$, where it equals 8.

The model holds the master, with first-stage capacity `Build` and the recourse estimate `theta`, and the scenario subproblem as named problems.
`Idle` makes `CapacityUse` an equality with two variables, so its dual is the multiplier of the
capacity right-hand side; the cut-validity section below shows what goes wrong with a
one-variable row. The run
switches problems with `ampl.eval("problem ...")`, because each named problem has its own option
environment (Named Problems and Environments (AMPL lab documentation, section 6.5.1)), reads `dual()` from the two subproblem rows, and appends the cut by item assignment:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set FIRST_STAGE;                      # first-stage capacity decisions
set SCENARIOS;                        # demand scenarios
param build_cost {FIRST_STAGE} >= 0;  # cost per unit of capacity
param max_build {FIRST_STAGE} >= 0;   # capacity limit, units
param probability {SCENARIOS} >= 0;          # scenario probability
param demand {SCENARIOS} >= 0;        # units demanded per scenario
param outsource_cost >= 0;            # cost per outsourced unit

# Master: first-stage capacity Build and the recourse estimate theta
var Build {i in FIRST_STAGE} >= 0, <= max_build[i];
var theta >= 0;                       # valid because recourse costs are nonnegative
var FirstStageCost = sum {i in FIRST_STAGE} build_cost[i] * Build[i];  # build cost of the master plan
minimize MasterObjective: FirstStageCost + theta;  # first-stage cost plus recourse estimate
param nOptCuts integer >= 0 default 0;  # optimality cuts added so far
param cut_constant {1..nOptCuts} default 0;  # cut intercept
param cut_coefficient {1..nOptCuts, FIRST_STAGE} default 0;  # cut slope per first-stage decision
subject to OptimalityCut {k in 1..nOptCuts}:  # theta at least every cut
    theta >= cut_constant[k]
             + sum {i in FIRST_STAGE} cut_coefficient[k,i] * Build[i];

# Subproblem: one scenario with the capacity fixed by the master
param scenario symbolic in SCENARIOS;
param capacity_fixed >= 0 default 0;  # master capacity Build, fixed here
var Produce >= 0;                     # units made in-house
var Idle >= 0;                        # unused capacity
var Outsource >= 0;                   # units bought outside
minimize RecourseCost: outsource_cost * Outsource;  # second-stage cost of one scenario
subject to Supply: Produce + Outsource >= demand[scenario];  # meet the scenario demand
subject to CapacityUse: Produce + Idle = capacity_fixed;  # its dual gives the cut slope

# master: first stage, theta and cuts
problem Master: Build, theta, FirstStageCost, MasterObjective, OptimalityCut;
# subproblem: one scenario at fixed capacity
problem Sub: Produce, Idle, Outsource, RecourseCost, Supply, CapacityUse;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["FIRST_STAGE"] = ["plant"]
ampl.set["SCENARIOS"] = ["low", "high"]
ampl.param["build_cost"] = {"plant": 1}
ampl.param["max_build"] = {"plant": 10}
ampl.param["probability"] = {"low": 0.5, "high": 0.5}
ampl.param["demand"] = {"low": 4, "high": 8}
ampl.param["outsource_cost"] = 3

In [5]:
# run.py
for problem in ["Master", "Sub"]:
    ampl.eval(f"problem {problem};")
    ampl.option["presolve"] = 0
probability = ampl.param["probability"].to_dict()
demand = ampl.param["demand"].to_dict()
upper = float("inf")
for iteration in range(1, 11):
    ampl.eval("problem Master;")
    ampl.solve(solver="highs", verbose=False)
    assert ampl.solve_result == "solved", ampl.solve_result
    lower = ampl.obj["MasterObjective"].value()
    x = ampl.var["Build"]["plant"].value()
    first_stage = ampl.var["FirstStageCost"].value()

    ampl.eval("problem Sub;")
    ampl.param["capacity_fixed"] = x
    recourse = constant = coefficient = 0
    for s in probability:
        ampl.param["scenario"] = s
        ampl.solve(solver="highs", verbose=False)
        assert ampl.solve_result == "solved", (s, ampl.solve_result)
        recourse += probability[s] * ampl.obj["RecourseCost"].value()
        constant += probability[s] * ampl.con["Supply"].dual() * demand[s]
        coefficient += probability[s] * ampl.con["CapacityUse"].dual()
    upper = min(upper, first_stage + recourse)
    print(f"Build = {x:g}  LB = {lower:g}  UB = {upper:g}"
          f"  cut: theta >= {constant:g} + {coefficient:g} Build")
    if upper - lower <= 1e-6:
        break
    k = int(ampl.param["nOptCuts"].value()) + 1
    ampl.param["nOptCuts"] = k
    ampl.param["cut_constant"] = {k: constant}
    ampl.param["cut_coefficient"] = {(k, "plant"): coefficient}

Build = 0  LB = 0  UB = 18  cut: theta >= 18 + -3 Build
Build = 6  LB = 6  UB = 9  cut: theta >= 12 + -1.5 Build
Build = 8  LB = 8  UB = 8  cut: theta >= 0 + 0 Build


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set FIRST_STAGE := plant;
set SCENARIOS := low high;
param build_cost := plant 1;
param max_build := plant 10;
param: probability demand :=
    low    0.5   4
    high   0.5   8 ;
param outsource_cost := 3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
problem Master; option solver highs, solver_msg 0, presolve 0;
problem Sub;    option solver highs, solver_msg 0, presolve 0;
param lower_bound;                   # master objective, a lower bound
param upper_bound default Infinity;  # best plan cost found, an upper bound
param recourse;                      # expected recourse at the current x
param constant;                      # cut intercept being accumulated
param coefficient;                   # cut slope being accumulated
for {iteration in 1..10} {
    solve Master;
    let lower_bound := MasterObjective;
    let capacity_fixed := Build['plant'];
    let recourse := 0;
    let constant := 0;
    let coefficient := 0;
    for {s in SCENARIOS} {
        let scenario := s;
        solve Sub;
        let recourse := recourse + probability[s] * RecourseCost;
        let constant := constant + probability[s] * Supply.dual * demand[s];
        let coefficient := coefficient + probability[s] * CapacityUse.dual;
    }
    let upper_bound := min(upper_bound, FirstStageCost + recourse);
    printf "Build = %g  LB = %g  UB = %g  cut: theta >= %g + %g Build\n",
        Build['plant'], lower_bound, upper_bound, constant, coefficient;
    if upper_bound - lower_bound <= 1e-6 then break;
    let nOptCuts := nOptCuts + 1;
    let cut_constant[nOptCuts] := constant;
    let cut_coefficient[nOptCuts, 'plant'] := coefficient;
}

Build = 0  LB = 0  UB = 18  cut: theta >= 18 + -3 Build
Build = 6  LB = 6  UB = 9  cut: theta >= 12 + -1.5 Build
Build = 8  LB = 8  UB = 8  cut: theta >= 0 + 0 Build
